install a package 

uv add sentence-transformers

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [3]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)


In [4]:
v1.dot(dv)

np.float32(0.32332397)

In [5]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [6]:
v2.dot(dv)

np.float32(0.019730574)

In [7]:
from ingest import load_faq_data

documents = load_faq_data()


In [8]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [9]:
from tqdm.auto import tqdm

In [10]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/25 [00:00<?, ?it/s]

1208

In [11]:
import numpy as np
X = np.array(vectors)

In [12]:
print(X.shape)

(1208, 384)


In [13]:
dv.shape

(384,)

In [14]:
query = 'Can I still join the course after the start date?'
v_query = model.encode(query)

In [15]:
scores = X.dot(v_query)


In [16]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(553), np.float32(0.762941))

In [17]:
documents[idx]


{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [18]:
top5 = np.argsort(scores)[-5:]
top5

array([558, 472,  29, 955, 553])

In [19]:
for idx in top5:
    print(documents[idx]['question'])
    print(documents[idx]['answer'])
    print()

Course - Can I follow the course after it finishes?
Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.

You can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

The course has already started. Can I still join it?
Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.

In order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.

Course - Can I still join the course 

In [21]:
top5 = np.argsort(-scores)[:5]
top5

array([553, 955,  29, 472, 558])